<a href="https://colab.research.google.com/github/mtharun9299/ML_PBL/blob/main/Crop_Disease_Detector_Colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crop Disease Detector for Low-End Phones
### MobileNetV2 + TensorFlow Lite (INT8 Quantization) + Grad-CAM Explainability

**Project-Based Learning (PBL) — Machine Learning (CS3505)**
Chennai Institute of Technology

This notebook is the full, runnable companion to the PBL report. It covers:
1. Setup & dataset loading (PlantVillage)
2. Data preprocessing & augmentation
3. **Iteration 1** — MobileNetV2 baseline (frozen backbone)
4. **Iteration 2** — Fine-tuning + augmentation refinement
5. Evaluation (accuracy, precision, recall, F1, confusion matrix)
6. TensorFlow Lite conversion with INT8 post-training quantization
7. Grad-CAM explainability (heatmap generation + overlay)
8. Exporting artifacts for the Android app (`.tflite` model + labels)

> Run cells top to bottom. Designed for Google Colab (free GPU runtime: *Runtime → Change runtime type → GPU*).


## 1. Setup

In [ ]:
# Install/verify dependencies (Colab usually has these preinstalled)
!pip install -q tensorflow opencv-python-headless matplotlib scikit-learn seaborn


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import cv2

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32


## 2. Dataset

This project uses the **PlantVillage** dataset. Two ways to get it in Colab:

**Option A — Kaggle (recommended):** upload your `kaggle.json` API token, then run the cell below.
**Option B — Manual:** mount Google Drive and point `DATASET_DIR` at your own copy of the dataset (folder-per-class layout).


In [ ]:
# --- Option A: Download via Kaggle API ---
# from google.colab import files
# files.upload()  # upload kaggle.json when prompted
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d emmarex/plantdisease -p /content/data --unzip

# --- Option B: Mount Google Drive ---
 from google.colab import drive
 drive.mount('/content/drive')
 DATASET_DIR = '/content/drive/MyDrive/PlantVillage'

# Set this to wherever your dataset folder (one sub-folder per class) lives:
DATASET_DIR = '/content/data/PlantVillage'

assert os.path.isdir(DATASET_DIR), (
    "Set DATASET_DIR to a folder containing one sub-folder per class "
    "(e.g. 'Tomato_Early_blight', 'Tomato_healthy', ...)."
)
CLASS_NAMES = sorted(os.listdir(DATASET_DIR))
NUM_CLASSES = len(CLASS_NAMES)
print(f"Found {NUM_CLASSES} classes:", CLASS_NAMES)


## 3. Data Preprocessing & Augmentation

In [ ]:
# Iteration 1 (baseline): rescale only, no augmentation
train_datagen_baseline = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15
)

# Iteration 2 (refined): augmentation added to improve robustness
train_datagen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.15,
    validation_split=0.15
)

def make_generators(datagen):
    train_gen = datagen.flow_from_directory(
        DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', subset='training', seed=SEED
    )
    val_gen = datagen.flow_from_directory(
        DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', subset='validation', seed=SEED
    )
    return train_gen, val_gen


## 4. Iteration 1 — Baseline (frozen MobileNetV2 backbone)

Matches Section 4.2 of the report: MobileNetV2 (ImageNet weights, frozen) + Global Average
Pooling + Dense(128, ReLU) + Softmax. No augmentation, 15 epochs.


In [ ]:
train_gen_1, val_gen_1 = make_generators(train_datagen_baseline)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
history_baseline = model.fit(
    train_gen_1,
    validation_data=val_gen_1,
    epochs=15
)


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history_baseline.history['accuracy'], label='train acc')
plt.plot(history_baseline.history['val_accuracy'], label='val acc')
plt.title('Iteration 1 (Baseline) — Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.show()


## 5. Iteration 2 — Refinement (augmentation + fine-tuning)

Matches Section 4.3: data augmentation, unfreeze the last few MobileNetV2 blocks, fine-tune
at a low learning rate, with class weighting for the mild class imbalance.


In [ ]:
train_gen_2, val_gen_2 = make_generators(train_datagen_aug)

# Unfreeze the last ~30 layers of the backbone for fine-tuning
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Class weights to correct mild imbalance
from sklearn.utils.class_weight import compute_class_weight
labels = train_gen_2.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_finetune = model.fit(
    train_gen_2,
    validation_data=val_gen_2,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history_finetune.history['accuracy'], label='train acc')
plt.plot(history_finetune.history['val_accuracy'], label='val acc')
plt.title('Iteration 2 (Refinement) — Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.show()


## 6. Evaluation — Accuracy, Precision, Recall, F1, Confusion Matrix

Matches Section 6.1–6.2 of the report.


In [ ]:
val_gen_2.reset()
y_true = val_gen_2.classes
y_pred_probs = model.predict(val_gen_2, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — Final Model (Iteration 2)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Results-across-iterations table (fill in with your actual run's numbers)
results_summary = {
    'Iteration 1 (baseline)': history_baseline.history['val_accuracy'][-1],
    'Iteration 2 (refined)':  history_finetune.history['val_accuracy'][-1],
}
for k, v in results_summary.items():
    print(f"{k}: {v*100:.1f}% validation accuracy")


## 7. TensorFlow Lite Conversion — INT8 Post-Training Quantization

Matches Section 4.5 / 5.2 of the report — compresses the model to fit low-end phones
(target: <4 MB, <3% accuracy drop).


In [ ]:
SAVED_MODEL_DIR = '/content/crop_disease_saved_model'
model.export(SAVED_MODEL_DIR)  # tf.keras 3.x — use model.save(...) with older TF if needed

converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset for full-integer quantization (recommended for best size/accuracy trade-off)
def representative_dataset():
    val_gen_2.reset()
    for _ in range(100):
        batch_x, _ = next(val_gen_2)
        for img in batch_x:
            yield [np.expand_dims(img, axis=0).astype(np.float32)]

converter.representative_dataset = representative_dataset

tflite_model = converter.convert()

TFLITE_PATH = '/content/crop_disease_model_quantized.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / (1024 * 1024)
print(f"Quantized model size: {size_mb:.2f} MB")


In [ ]:
# Sanity-check the quantized model with the TFLite interpreter
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input:", input_details[0]['shape'], input_details[0]['dtype'])
print("Output:", output_details[0]['shape'], output_details[0]['dtype'])

# Export class labels for the Android app
LABELS_PATH = '/content/labels.txt'
with open(LABELS_PATH, 'w') as f:
    f.write('\n'.join(CLASS_NAMES))
print("Saved labels to", LABELS_PATH)


## 8. Grad-CAM — Explainable AI Layer

Matches Section 4.4 / 5.2 of the report. Because MobileNetV2 ends in Global Average
Pooling before the Dense head, this reduces to a single forward pass at inference time
on-device — the code below (using `GradientTape`) is the *training-side* reference
implementation used to validate the heatmaps before porting the CAM logic to Android.


In [ ]:
LAST_CONV_LAYER_NAME = 'out_relu'  # final MobileNetV2 conv layer

def generate_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)


def overlay_heatmap(img_path, heatmap, alpha=0.4):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0), img


In [ ]:
# Try it on a sample validation image
sample_path = val_gen_2.filepaths[0]

img = tf.keras.preprocessing.image.load_img(sample_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

heatmap, pred_idx = generate_gradcam_heatmap(img_array, model, LAST_CONV_LAYER_NAME)
overlay, original = overlay_heatmap(sample_path, heatmap)

print("Predicted class:", CLASS_NAMES[pred_idx])

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(original); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(heatmap, cmap='jet'); ax[1].set_title('Grad-CAM Heatmap'); ax[1].axis('off')
ax[2].imshow(overlay); ax[2].set_title(f'Overlay: {CLASS_NAMES[pred_idx]}'); ax[2].axis('off')
plt.tight_layout()
plt.show()


## 9. Dual-Output Model Export (on-device CAM, near-zero overhead)

Matches the "TFLite Dual-Output Graph Export" approach from the report: export a model
with **two** outputs — class probabilities and the final conv-layer feature map — so the
Android app can compute the CAM heatmap with one forward pass and a dot product,
with no gradients needed on-device.


In [ ]:
last_conv_layer = model.get_layer(LAST_CONV_LAYER_NAME)

cam_model = Model(
    inputs=model.input,
    outputs=[model.output, last_conv_layer.output]
)

CAM_SAVED_MODEL_DIR = '/content/crop_disease_cam_saved_model'
cam_model.export(CAM_SAVED_MODEL_DIR)

cam_converter = tf.lite.TFLiteConverter.from_saved_model(CAM_SAVED_MODEL_DIR)
cam_converter.optimizations = [tf.lite.Optimize.DEFAULT]
cam_tflite_model = cam_converter.convert()

CAM_TFLITE_PATH = '/content/crop_detector_with_xai.tflite'
with open(CAM_TFLITE_PATH, 'wb') as f:
    f.write(cam_tflite_model)

print("Dual-output (XAI-ready) model size:",
      f"{os.path.getsize(CAM_TFLITE_PATH) / (1024*1024):.2f} MB")


## 10. Download Artifacts for the Android App

Download these three files and drop them into the Android Studio project's `app/src/main/assets/` folder:
- `crop_disease_model_quantized.tflite` — classification-only model (smaller, faster)
- `crop_detector_with_xai.tflite` — dual-output model (predictions + feature map for on-device Grad-CAM/CAM)
- `labels.txt` — class name list, in the same order as the model's output indices


In [ ]:
from google.colab import files

files.download(TFLITE_PATH)
files.download(CAM_TFLITE_PATH)
files.download(LABELS_PATH)


## 11. Android Integration Notes (reference only — Kotlin/Java lives in the Android Studio project)

- Load `crop_detector_with_xai.tflite` with `org.tensorflow.lite.Interpreter`.
- Run `interpreter.runForMultipleInputsOutputs(...)` to get both the class-probability
  tensor and the `7x7x1280` feature-map tensor in a single pass.
- Take the predicted class's row of Dense-layer weights (exported alongside the model, or
  read via `interpreter.getOutputTensor(1)`), dot-product it with the feature map, apply
  ReLU, normalize to `[0,1]`, resize to `224x224`, and render as a semi-transparent
  `COLORMAP_JET`-style overlay using Android's `Canvas`/`Bitmap` APIs — this mirrors the
  `overlay_heatmap()` function above.
- Store the diagnosis-to-treatment mapping in a local SQLite database (`SQLiteOpenHelper`)
  bundled with the app so recommendations work fully offline.
